In [5]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.genereux_uncertainty_propagation as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

#wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']
wade_tracers = ['Ca_mg_L', 'Mg_mg_L', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_febros_fractions_df,
    wade_febros_scaler,
    wade_febros_pca,
    wade_febros_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-02-14 00:00:00",
    end_date="2023-02-20 00:00:00",
    endmember_ids=["RI23-5018", "RI23-5000", "RI23-5005"],
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5018", "RI23-5000", "RI23-5005"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-02-14 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-02-20 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# Define confidence levels to process
confidence_levels = [0.70, 0.95]
results_dict = {}

for conf in confidence_levels:
    # 5. Calculate Genereux Uncertainties for the given confidence level
    uncertainty_df = up.propagate_genereux_uncertainty(
        stream_df=stream_event_df,
        em_grouped=wade_febros_endmembers_df,
        em_raw=em_raw_subset,
        tracers=wade_tracers,
        analytical_sd=analytical_sd,
        confidence_level=conf
    )

    # 6. Merge fractions and their calculated uncertainties
    results_with_error = pd.merge(
        wade_febros_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
    )

    # Sort chronologically by Datetime and reset index
    results_with_error = results_with_error.sort_values("Datetime").reset_index(drop=True)

    # Store in a dictionary for easy access in your notebook/script
    results_dict[conf] = results_with_error

    # Define dynamic filename and save to CSV in the output directory
    conf_pct = int(conf * 100)
    output_filename = output_dir / f"Wade_Feb_ROS_fractions_with_uncertainty_{conf_pct}pct.csv"
    results_with_error.to_csv(output_filename, index=False)
    
    print(f"✅ Successfully saved {conf_pct}% CI results to: {output_filename}")

# Optional: Access individual dataframes if needed later in your session
results_with_error_70 = results_dict[0.70]
results_with_error_95 = results_dict[0.95]

# Preview the sorted 95% confidence results head
display_cols = ["Sample ID", "Datetime"]
uncertainty_cols_95 = [col for col in results_with_error_95.columns if "Uncertainty_95sig" in col]
fraction_cols = [col.replace("_Uncertainty_95sig", "") for col in uncertainty_cols_95]
for frac, unc in zip(fraction_cols, uncertainty_cols_95):
    display_cols.extend([frac, unc])

print("\nPreview of Chronologically Sorted 95% CI Results:")
print(results_with_error_95[display_cols].head())

✅ Successfully saved 70% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Wade_Feb_ROS_fractions_with_uncertainty_70pct.csv
✅ Successfully saved 95% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Wade_Feb_ROS_fractions_with_uncertainty_95pct.csv

Preview of Chronologically Sorted 95% CI Results:
   Sample ID            Datetime  Groundwater  Groundwater_Uncertainty_95sig  \
0  RI23-1025 2023-02-15 12:00:00     0.429392                       0.758162   
1  RI23-1009 2023-02-15 15:00:00     0.499577                       0.870153   
2  RI23-1010 2023-02-15 19:00:00     0.450879                       0.759665   
3  RI23-1011 2023-02-15 23:00:00     0.414333                       0.722776   
4  RI23-1012 2023-02-16 03:00:00     0.245741                       0.597952   

   Snowmelt lysimeter  Snowmelt lysimeter_Uncertainty_95sig  \
0            0.465132                              1.6974

In [6]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.genereux_uncertainty_propagation as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

#wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']
wade_tracers = ['Ca_mg_L', 'Mg_mg_L', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_martherm_fractions_df,
    wade_martherm_scaler,
    wade_martherm_pca,
    wade_martherm_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-03-21 00:00:00",
    end_date="2023-03-26 00:00:00",
    endmember_ids=[
                   "RI23-5018", #"RI23-5006", # Homeowner well groundwater March 2023
                   #"RI23-1034", # Pre-event baseflow, labeled as GW in Wade index
                   "RI23-5005", # Snowmelt lysimeter 02/15/2023
                   #"RI23-1063", # Snowmelt lysimeter 3/28
                   "RI23-5009", # Soil water lysimeter march
                   ], 
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5018", "RI23-5005", "RI23-5009"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-21 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-03-26 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# Define confidence levels to process
confidence_levels = [0.70, 0.95]
results_dict = {}

for conf in confidence_levels:
    # 5. Calculate Genereux Uncertainties for the given confidence level
    uncertainty_df = up.propagate_genereux_uncertainty(
        stream_df=stream_event_df,
        em_grouped=wade_martherm_endmembers_df,
        em_raw=em_raw_subset,
        tracers=wade_tracers,
        analytical_sd=analytical_sd,
        confidence_level=conf
    )

    # 6. Merge fractions and their calculated uncertainties
    results_with_error = pd.merge(
        wade_martherm_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
    )

    # Sort chronologically by Datetime and reset index
    results_with_error = results_with_error.sort_values("Datetime").reset_index(drop=True)

    # Store in a dictionary for easy access in your notebook/script
    results_dict[conf] = results_with_error

    # Define dynamic filename and save to CSV in the output directory
    conf_pct = int(conf * 100)
    output_filename = output_dir / f"Wade_Mar_them_fractions_with_uncertainty_{conf_pct}pct.csv"
    results_with_error.to_csv(output_filename, index=False)
    
    print(f"✅ Successfully saved {conf_pct}% CI results to: {output_filename}")

# Optional: Access individual dataframes if needed later in your session
results_with_error_70 = results_dict[0.70]
results_with_error_95 = results_dict[0.95]

# Preview the sorted 95% confidence results head
display_cols = ["Sample ID", "Datetime"]
uncertainty_cols_95 = [col for col in results_with_error_95.columns if "Uncertainty_95sig" in col]
fraction_cols = [col.replace("_Uncertainty_95sig", "") for col in uncertainty_cols_95]
for frac, unc in zip(fraction_cols, uncertainty_cols_95):
    display_cols.extend([frac, unc])

print("\nPreview of Chronologically Sorted 95% CI Results:")
print(results_with_error_95[display_cols].head())

✅ Successfully saved 70% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Wade_Mar_them_fractions_with_uncertainty_70pct.csv
✅ Successfully saved 95% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Wade_Mar_them_fractions_with_uncertainty_95pct.csv

Preview of Chronologically Sorted 95% CI Results:
   Sample ID            Datetime  Groundwater  Groundwater_Uncertainty_95sig  \
0  RI23-1039 2023-03-22 18:00:00     0.554301                       0.867697   
1  RI23-1040 2023-03-23 00:00:00     0.492368                       0.833870   
2  RI23-1041 2023-03-23 06:00:00     0.504608                       0.830502   
3  RI23-1055 2023-03-23 12:00:00     0.493162                       0.809555   
4  RI23-1056 2023-03-23 18:00:00     0.324627                       0.640867   

   Snowmelt lysimeter  Snowmelt lysimeter_Uncertainty_95sig  \
0            0.445699                              0.86

In [7]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.genereux_uncertainty_propagation as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

#wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']
wade_tracers = ['Ca_mg_L', 'Mg_mg_L', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_fmelt_fractions_df,
    wade_fmelt_scaler,
    wade_fmelt_pca,
    wade_fmelt_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-03-30 00:00:00",
    end_date="2023-04-15 00:00:00",
    endmember_ids=[
                   "RI23-5018", #"RI23-5006", # Homeowner well groundwater March 2023
                   #"RI22-0860", "RI22-0859", # Homeowner well groundwater 2022
                   #"RI23-1034", # Pre-event baseflow, labeled as GW in Wade index
                   #"RI25-1111", "RI25-1126", "RI25-1183", "RI25-1201", "RI25-1290", # 2025 baseflow samples 
                   #"RI23-5005", # Snowmelt lysimeter 02/15/2023
                   #"RI23-1063", # Snowmelt lysimeter 3/28
                   "RI23-1098", # Snowmelt lysimeter 04/11/2023
                   #"RI23-5009", # Soil water lysimeter sample
                   "RI23-5011", # Soil water lysimeter wet 4/12
                   ],  
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5018", "RI23-1098", "RI23-5011"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-30 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-04-15 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# Define confidence levels to process
confidence_levels = [0.70, 0.95]
results_dict = {}

for conf in confidence_levels:
    # 5. Calculate Genereux Uncertainties for the given confidence level
    uncertainty_df = up.propagate_genereux_uncertainty(
        stream_df=stream_event_df,
        em_grouped=wade_fmelt_endmembers_df,
        em_raw=em_raw_subset,
        tracers=wade_tracers,
        analytical_sd=analytical_sd,
        confidence_level=conf
    )

    # 6. Merge fractions and their calculated uncertainties
    results_with_error = pd.merge(
        wade_fmelt_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
    )

    # Sort chronologically by Datetime and reset index
    results_with_error = results_with_error.sort_values("Datetime").reset_index(drop=True)

    # Store in a dictionary for easy access in your notebook/script
    results_dict[conf] = results_with_error

    # Define dynamic filename and save to CSV in the output directory
    conf_pct = int(conf * 100)
    output_filename = output_dir / f"Wade_final_melt_fractions_with_uncertainty_{conf_pct}pct.csv"
    results_with_error.to_csv(output_filename, index=False)
    
    print(f"✅ Successfully saved {conf_pct}% CI results to: {output_filename}")

# Optional: Access individual dataframes if needed later in your session
results_with_error_70 = results_dict[0.70]
results_with_error_95 = results_dict[0.95]

# Preview the sorted 95% confidence results head
display_cols = ["Sample ID", "Datetime"]
uncertainty_cols_95 = [col for col in results_with_error_95.columns if "Uncertainty_95sig" in col]
fraction_cols = [col.replace("_Uncertainty_95sig", "") for col in uncertainty_cols_95]
for frac, unc in zip(fraction_cols, uncertainty_cols_95):
    display_cols.extend([frac, unc])

print("\nPreview of Chronologically Sorted 95% CI Results:")
print(results_with_error_95[display_cols].head())

✅ Successfully saved 70% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Wade_final_melt_fractions_with_uncertainty_70pct.csv
✅ Successfully saved 95% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Wade_final_melt_fractions_with_uncertainty_95pct.csv

Preview of Chronologically Sorted 95% CI Results:
   Sample ID            Datetime  Groundwater  Groundwater_Uncertainty_95sig  \
0  RI23-1064 2023-03-31 08:00:00     0.500580                       1.405926   
1  RI23-1065 2023-03-31 14:00:00     0.374702                       1.188137   
2  RI23-1066 2023-03-31 20:00:00     0.457860                       1.293460   
3  RI23-1067 2023-04-01 02:00:00     0.455074                       1.347334   
4  RI23-1068 2023-04-01 08:00:00     0.481835                       1.387673   

   Snowmelt lysimeter  Snowmelt lysimeter_Uncertainty_95sig  \
0        0.000000e+00                              